In [90]:
# %%
# CORRIGIR NUMPY PRIMEIRO
import subprocess
import sys

print("Corrigindo versao do NumPy...")
subprocess.check_call([sys.executable, "-m", "pip",
                      "install", "numpy<2", "--quiet"])
print("NumPy corrigido. REINICIE O KERNEL agora!")

Corrigindo versao do NumPy...
NumPy corrigido. REINICIE O KERNEL agora!


In [91]:
from pyspark.sql import SparkSession

# Configurações otimizadas para Spark
spark = (
    SparkSession.builder
    .appName("oracle-connection-check")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryoserializer.buffer.max", "512m")
    .getOrCreate()
)

print("OK Spark Session criada com otimizações")

OK Spark Session criada com otimizações


## Configuracao Oracle

Ajuste usuario e senha.


In [92]:
oracle_host = "10.255.150.11"
oracle_port = 1521
oracle_service = "bi.grupotracker.com.br"

oracle_user = "clickhouse"
oracle_password = "qiU!EOoe"

jdbc_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service}"
jdbc_url

'jdbc:oracle:thin:@//10.255.150.11:1521/bi.grupotracker.com.br'

## Teste rapido de conectividade TCP (opcional)

Se falhar aqui, nao adianta tentar JDBC.


In [94]:
import socket

def check_tcp(host: str, port: int, timeout: float = 5.0) -> bool:
 sock = socket.socket()
 sock.settimeout(timeout)
 try:
 sock.connect((host, int(port)))
 print(f"OK: {host}:{port}")
 return True
 except Exception as e:
 print(f"FAIL: {host}:{port} -> {e}")
 return False
 finally:
 sock.close()

check_tcp(oracle_host, oracle_port)


IndentationError: expected an indented block after 'try' statement on line 6 (2835459250.py, line 7)

## Leitura simples no Oracle

Usa `query` para validar que o JDBC funciona.


In [ ]:
df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("user", oracle_user)
    .option("password", oracle_password)
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("query", "SELECT 1 AS ok FROM dual")
    .load()
)

df.show()

## Pipeline de extracao (11 tabelas)

O objetivo aqui e apenas validar leitura. Para cada tabela, fazemos um `show(5)`.


In [ ]:
tables_to_extract = [
 "ginf.depara_cliente",
 "ginf.BASE_CEP_COMPLETA",
 "ginf.TST_CONTRATOS_BI",
 "siga.SC5030",
 "siga.SC6030",
 "ginf.TST_HISTORICO_SOLICITACOES",
 "ginf.TST_SOLICIT_CADASTRADAS",
 "siga.SD2030",
 "siga.SF2030",
 "siga.ZTX030",
 "ginf.TST_CONTRATOS",
]

print(f"Total de tabelas para extrair: {len(tables_to_extract)}")
for t in tables_to_extract:
 print(f" - {t}")

# %%
import clickhouse_connect
from datetime import datetime

# ClickHouse configuration
CLICKHOUSE_CONFIG = {
 'host': "e1a1lieug8.us-central1.gcp.clickhouse.cloud",
 'port': 8443,
 'username': "default",
 'password': "_uv765EvWphL_",
 'database': "default",
 'secure': True,
 'connect_timeout': 60,
 'send_receive_timeout': 300
}


def build_jdbc_options(query):
 """Build JDBC connection options"""
 return {
 "url": jdbc_url,
 "user": oracle_user,
 "password": oracle_password,
 "driver": "oracle.jdbc.OracleDriver",
 "dbtable": query
 }


def read_jdbc_query(query):
 """Execute JDBC query and return DataFrame"""
 options = build_jdbc_options(query)
 return spark.read.format("jdbc").options(**options).load()


def create_clickhouse_client():
 """Create ClickHouse client"""
 client = clickhouse_connect.get_client(**CLICKHOUSE_CONFIG)
 client.command('SELECT 1')
 print("Conexao com ClickHouse estabelecida\n")
 return client


def write_to_clickhouse(spark_df, oracle_table, clickhouse_table, client):
 """Write DataFrame to ClickHouse"""
 try:
 start = datetime.now()
 pandas_df = spark_df.toPandas()
 row_count = len(pandas_df)

 client.insert_df(clickhouse_table, pandas_df)

 duration = (datetime.now() - start).total_seconds()
 msg = f" [OK] {row_count:,} linhas em {duration:.2f}s -> {clickhouse_table}"
 print(msg)
 return True, row_count

 except Exception as e:
 error_msg = str(e)[:120]
 print(f" [ERRO] {error_msg}")
 return False, 0


def extract_table_name(full_name):
 """Extract table name from schema.table"""
 return full_name.split('.')[-1].lower()


# Main execution
separator = "=" * 60
print(separator)
print("MIGRACAO ORACLE -> CLICKHOUSE")
print(separator)

start_total = datetime.now()
client = create_clickhouse_client()

success_count = 0
failed_count = 0
total_rows = 0
failed_tables = []

for idx, oracle_table in enumerate(tables_to_extract, 1):
 pct = (idx / len(tables_to_extract)) * 100
 print(f"[{idx}/{len(tables_to_extract)}] ({pct:.1f}%) {oracle_table}")

 try:
 query = f"(SELECT * FROM {oracle_table}) tmp"
 df = read_jdbc_query(query)
 ch_table = extract_table_name(oracle_table)

 ok, rows = write_to_clickhouse(df, oracle_table, ch_table, client)

 if ok:
 success_count += 1
 total_rows += rows
 else:
 failed_count += 1
 failed_tables.append(oracle_table)

 except Exception as e:
 error_msg = str(e)[:120]
 print(f" [ERRO] {error_msg}")
 failed_count += 1
 failed_tables.append(oracle_table)

 print()

duration_total = (datetime.now() - start_total).total_seconds()

print(separator)
print("RESUMO")
print(separator)
print(f"Sucesso: {success_count}/{len(tables_to_extract)}")
print(f"Falhas: {failed_count}")
print(f"Total de linhas: {total_rows:,}")
print(f"Tempo: {duration_total:.2f}s")

if failed_tables:
 print("\nTabelas com erro:")
 for t in failed_tables:
 print(f" - {t}")

print(separator)


Preciso ler o notebook para entender melhor a estrutura das tabelas Oracle e criar os schemas correspondentes no ClickHouse.

<llm-progress-log-group></llm-progress-log-group>

In [ ]:
# %% PASSO 1 - Verificar NumPy
import numpy as np

print(f"NumPy versao: {np.__version__}")
# Se mostrar 2.x.x, precisa corrigir

In [ ]:
# %% Verificar NumPy
import numpy as np

print(f"NumPy: {np.__version__}")
# Deve mostrar 1.26.4

In [ ]:
!pip install "numpy<2" --force-reinstall
print("AGORA REINICIE O KERNEL! (Kernel > Restart Kernel)")

In [ ]:
# %%
import clickhouse_connect
from datetime import datetime

# Configuracao ClickHouse
client = clickhouse_connect.get_client(
 host="e1a1lieug8.us-central1.gcp.clickhouse.cloud",
 port=8443,
 username="default",
 password="_uv765EvWphL_",
 database="default",
 secure=True,
 connect_timeout=60,
 send_receive_timeout=300
)

print("Conexao ClickHouse: OK\n")

# JDBC config
jdbc_opts = {
 "url": jdbc_url,
 "user": oracle_user,
 "password": oracle_password,
 "driver": "oracle.jdbc.OracleDriver"
}

sep = "=" * 60
print(sep)
print("MIGRACAO ORACLE -> CLICKHOUSE")
print(sep)

start = datetime.now()
ok_count = 0
fail_count = 0
total_rows = 0
failed = []

for i, tbl in enumerate(tables_to_extract, 1):
 pct = (i / len(tables_to_extract)) * 100
 ch_tbl = tbl.split('.')[-1].lower()

 print(f"[{i}/{len(tables_to_extract)}] ({pct:.1f}%) {tbl}")

 try:
 t0 = datetime.now()

 # Ler Oracle
 df = spark.read.format("jdbc") \
 .options(**jdbc_opts) \
 .option("dbtable", f"(SELECT * FROM {tbl}) tmp") \
 .load()

 # Converter e inserir
 pdf = df.toPandas()
 rows = len(pdf)
 client.insert_df(ch_tbl, pdf)

 dur = (datetime.now() - t0).total_seconds()
 print(f" [OK] {rows:,} linhas em {dur:.2f}s -> {ch_tbl}")

 ok_count += 1
 total_rows += rows

 except Exception as e:
 msg = str(e)[:150]
 print(f" [ERRO] {msg}")
 fail_count += 1
 failed.append(tbl)

 print()

dur_total = (datetime.now() - start).total_seconds()

print(sep)
print("RESUMO")
print(sep)
print(f"Sucesso: {ok_count}/{len(tables_to_extract)}")
print(f"Falhas: {fail_count}")
print(f"Total linhas: {total_rows:,}")
print(f"Tempo: {dur_total:.2f}s")

if failed:
 print("\nFalhas:")
 for t in failed:
 print(f" - {t}")

print(sep)


In [ ]:
# %%
import clickhouse_connect
from datetime import datetime

# Configuracao ClickHouse
CLICKHOUSE_CONFIG = {
 'host': "e1a1lieug8.us-central1.gcp.clickhouse.cloud",
 'port': 8443,
 'username': "default",
 'password': "_uv765EvWphL_",
 'database': "default",
 'secure': True,
 'connect_timeout': 60,
 'send_receive_timeout': 300
}

# Criar cliente ClickHouse
client = clickhouse_connect.get_client(**CLICKHOUSE_CONFIG)
client.command('SELECT 1')
print("Conexao ClickHouse: OK\n")

# Configuracao JDBC
jdbc_options = {
 "url": jdbc_url,
 "user": oracle_user,
 "password": oracle_password,
 "driver": "oracle.jdbc.OracleDriver"
}

separator = "=" * 60
print(separator)
print("MIGRACAO ORACLE -> CLICKHOUSE")
print(separator)

start_total = datetime.now()
success = 0
failed = 0
total_rows = 0
failed_tables = []

for idx, oracle_table in enumerate(tables_to_extract, 1):
 pct = (idx / len(tables_to_extract)) * 100
 ch_table = oracle_table.split('.')[-1].lower()

 print(f"[{idx}/{len(tables_to_extract)}] ({pct:.1f}%) {oracle_table}")

 try:
 start = datetime.now()

 # Ler do Oracle
 query = f"(SELECT * FROM {oracle_table}) tmp"
 df = spark.read.format("jdbc") \
 .options(**jdbc_options) \
 .option("dbtable", query) \
 .load()

 # Coletar sample para verificar
 row_count = df.count()

 # Converter para Pandas em chunks menores se necessario
 if row_count > 100000:
 print(f" Tabela grande ({row_count:,} linhas), processando em lotes...")
 # Para tabelas grandes, use batch
 batch_size = 50000
 for i in range(0, row_count, batch_size):
 batch_df = df.limit(batch_size).offset(i).toPandas()
 if i == 0:
 client.insert_df(ch_table, batch_df)
 else:
 client.insert_df(ch_table, batch_df)
 else:
 pandas_df = df.toPandas()
 client.insert_df(ch_table, pandas_df)

 duration = (datetime.now() - start).total_seconds()
 print(f" [OK] {row_count:,} linhas em {duration:.2f}s -> {ch_table}")

 success += 1
 total_rows += row_count

 except Exception as e:
 error_msg = str(e)[:150]
 print(f" [ERRO] {error_msg}")
 failed += 1
 failed_tables.append(oracle_table)

 print()

duration_total = (datetime.now() - start_total).total_seconds()

print(separator)
print("RESUMO")
print(separator)
print(f"Sucesso: {success}/{len(tables_to_extract)}")
print(f"Falhas: {failed}")
print(f"Total de linhas: {total_rows:,}")
print(f"Tempo: {duration_total:.2f}s")

if failed_tables:
 print("\nTabelas com erro:")
 for t in failed_tables:
 print(f" - {t}")

print(separator)


In [ ]:
# %%
import clickhouse_connect
from datetime import datetime

# Configuracao ClickHouse
client = clickhouse_connect.get_client(
 host="e1a1lieug8.us-central1.gcp.clickhouse.cloud",
 port=8443,
 username="default",
 password="_uv765EvWphL_",
 database="default",
 secure=True,
 connect_timeout=60,
 send_receive_timeout=300
)

print("Conexao ClickHouse: OK\n")

# JDBC config
jdbc_opts = {
 "url": jdbc_url,
 "user": oracle_user,
 "password": oracle_password,
 "driver": "oracle.jdbc.OracleDriver"
}

sep = "=" * 60
print(sep)
print("MIGRACAO ORACLE -> CLICKHOUSE")
print(sep)

start = datetime.now()
ok_count = 0
fail_count = 0
total_rows = 0
failed = []

for i, tbl in enumerate(tables_to_extract, 1):
 pct = (i / len(tables_to_extract)) * 100
 ch_tbl = tbl.split('.')[-1].lower()

 print(f"[{i}/{len(tables_to_extract)}] ({pct:.1f}%) {tbl}")

 try:
 t0 = datetime.now()

 # Ler Oracle
 df = spark.read.format("jdbc") \
 .options(**jdbc_opts) \
 .option("dbtable", f"(SELECT * FROM {tbl}) tmp") \
 .load()

 # Converter e inserir
 pdf = df.toPandas()
 rows = len(pdf)
 client.insert_df(ch_tbl, pdf)

 dur = (datetime.now() - t0).total_seconds()
 print(f" [OK] {rows:,} linhas em {dur:.2f}s -> {ch_tbl}")

 ok_count += 1
 total_rows += rows

 except Exception as e:
 msg = str(e)[:150]
 print(f" [ERRO] {msg}")
 fail_count += 1
 failed.append(tbl)

 print()

dur_total = (datetime.now() - start).total_seconds()

print(sep)
print("RESUMO")
print(sep)
print(f"Sucesso: {ok_count}/{len(tables_to_extract)}")
print(f"Falhas: {fail_count}")
print(f"Total linhas: {total_rows:,}")
print(f"Tempo: {dur_total:.2f}s")

if failed:
 print("\nFalhas:")
 for t in failed:
 print(f" - {t}")

print(sep)


In [ ]:
# %%
import clickhouse_connect
from datetime import datetime

# ClickHouse connection configuration
CLICKHOUSE_CONFIG = {
 'host': "e1a1lieug8.us-central1.gcp.clickhouse.cloud",
 'port': 8443, # Use HTTPS port for cloud
 'username': "default",
 'password': "_uv765EvWphL_",
 'database': "default",
 'secure': True, # Enable SSL for cloud connection
 'connect_timeout': 60, # Increase timeout for cloud connections
 'send_receive_timeout': 300
}


def create_clickhouse_client():
 """Create and validate ClickHouse client connection"""
 try:
 client = clickhouse_connect.get_client(**CLICKHOUSE_CONFIG)
 # Validate connection
 client.command('SELECT 1')
 print("Conexao com ClickHouse estabelecida")
 return client
 except Exception as error:
 print(f"ERRO ao conectar ao ClickHouse: {str(error)}")
 raise


def write_to_clickhouse(spark_df, oracle_table_name, clickhouse_table_name, client) -> bool:
 """Write Spark DataFrame to ClickHouse table"""
 try:
 pandas_df = spark_df.toPandas()
 client.insert_df(clickhouse_table_name, pandas_df)

 row_count = len(pandas_df)
 print(f"[OK] {oracle_table_name}: {row_count:,} linhas gravadas em {clickhouse_table_name}")
 return True

 except Exception as error:
 error_message = str(error)[:100]
 print(f"[ERRO] Falha ao gravar {oracle_table_name}: {error_message}")
 return False


def extract_table_name(full_table_name: str) -> str:
 """Extract table name from schema.table format"""
 return full_table_name.split('.')[-1].lower()


def process_oracle_to_clickhouse(oracle_tables: list, client) -> dict:
 """Process and migrate tables from Oracle to ClickHouse"""
 results = {'success': 0, 'failed': 0, 'failed_tables': []}
 total_tables = len(oracle_tables)

 print(f"\nMIGRACAO ORACLE -> CLICKHOUSE")
 print(f"{'=' * 60}")
 print(f"Total de tabelas: {total_tables}\n")

 for index, oracle_table in enumerate(oracle_tables, 1):
 progress_pct = (index / total_tables) * 100
 print(f"[{index}/{total_tables}] ({progress_pct:.1f}%) Processando {oracle_table}...")

 try:
 df_oracle = read_jdbc_query(f"(SELECT * FROM {oracle_table}) tmp")
 clickhouse_table = extract_table_name(oracle_table)

 if write_to_clickhouse(df_oracle, oracle_table, clickhouse_table, client):
 results['success'] += 1
 else:
 results['failed'] += 1
 results['failed_tables'].append(oracle_table)

 except Exception as error:
 error_message = str(error)[:100]
 print(f"[ERRO] Falha ao processar {oracle_table}: {error_message}")
 results['failed'] += 1
 results['failed_tables'].append(oracle_table)

 print() # Linha em branco para separar tabelas

 return results


# Main execution
start_migration = datetime.now()
client = create_clickhouse_client()
migration_results = process_oracle_to_clickhouse(tables_to_extract, client)
duration_migration = (datetime.now() - start_migration).total_seconds()

print(f"{'=' * 60}")
print("RESUMO DA MIGRACAO")
print(f"{'=' * 60}")
print(f"Tabelas migradas com sucesso: {migration_results['success']}")
print(f"Tabelas com erro: {migration_results['failed']}")
print(f"Tempo total: {duration_migration:.2f}s")
if migration_results['failed_tables']:
 print(f"\nTabelas que falharam:")
 for table in migration_results['failed_tables']:
 print(f" - {table}")
print(f"{'=' * 60}")

In [ ]:
# %%
import clickhouse_connect
from typing import Optional

# ClickHouse connection configuration
CLICKHOUSE_CONFIG = {
 'host': "e1a1lieug8.us-central1.gcp.clickhouse.cloud",
 'port': 8443, # Use HTTPS port for cloud
 'username': "default",
 'password': "_uv765EvWphL_",
 'database': "default",
 'secure': True, # Enable SSL for cloud connection
 'connect_timeout': 60, # Increase timeout for cloud connections
 'send_receive_timeout': 300
}


def create_clickhouse_client():
 """Create and validate ClickHouse client connection"""
 try:
 client = clickhouse_connect.get_client(**CLICKHOUSE_CONFIG)
 # Validate connection
 client.command('SELECT 1')
 print("Conexao com ClickHouse estabelecida")
 return client
 except Exception as error:
 print(f"ERRO ao conectar ao ClickHouse: {str(error)}")
 raise


def write_to_clickhouse(spark_df, oracle_table_name, clickhouse_table_name, client) -> bool:
 """Write Spark DataFrame to ClickHouse table"""
 try:
 pandas_df = spark_df.toPandas()
 client.insert_df(clickhouse_table_name, pandas_df)

 row_count = len(pandas_df)
 print(f"[OK] {oracle_table_name}: {row_count:,} linhas gravadas em {clickhouse_table_name}")
 return True

 except Exception as error:
 error_message = str(error)[:100]
 print(f"[ERRO] Falha ao gravar {oracle_table_name}: {error_message}")
 return False


def extract_table_name(full_table_name: str) -> str:
 """Extract table name from schema.table format"""
 return full_table_name.split('.')[-1].lower()


def process_oracle_to_clickhouse(oracle_tables: list, client) -> dict:
 """Process and migrate tables from Oracle to ClickHouse"""
 results = {'success': 0, 'failed': 0, 'failed_tables': []}

 for oracle_table in oracle_tables:
 try:
 df_oracle = read_jdbc_query(f"(SELECT * FROM {oracle_table}) tmp")
 clickhouse_table = extract_table_name(oracle_table)

 if write_to_clickhouse(df_oracle, oracle_table, clickhouse_table, client):
 results['success'] += 1
 else:
 results['failed'] += 1
 results['failed_tables'].append(oracle_table)

 except Exception as error:
 error_message = str(error)[:100]
 print(f"[ERRO] Falha ao processar {oracle_table}: {error_message}")
 results['failed'] += 1
 results['failed_tables'].append(oracle_table)

 return results


# Main execution
client = create_clickhouse_client()
migration_results = process_oracle_to_clickhouse(tables_to_extract, client)

print(f"\n{'=' * 60}")
print("RESUMO DA MIGRACAO")
print(f"{'=' * 60}")
print(f"Tabelas migradas com sucesso: {migration_results['success']}")
print(f"Tabelas com erro: {migration_results['failed']}")
if migration_results['failed_tables']:
 print(f"Tabelas que falharam: {', '.join(migration_results['failed_tables'])}")
print(f"{'=' * 60}")